In [ ]:
import pandas as pd
import numpy as np


from dataset_evaluation.evaluation_framework import EvaluationFramework
from dataset_evaluation.new_dataset_evaluation import *

from pathlib import Path
from collections import defaultdict
from tqdm.notebook import tqdm

import matplotlib.pyplot as plt
plt.style.use("seaborn-v0_8-whitegrid")

import ast

from datetime import datetime

from dataset_evaluation.modelasametric.simlex999_comparison import (
    pairwise_spearman,
    select_invocab_simlex,
    compute_similarity,
    train_and_save_w2v
)
from scipy.stats import spearmanr

## Load datasets

### New dataset

In [ ]:
path = Path(None)

df_generated_stories = pd.concat(
	(pd.read_csv(file) for file in path.glob("stories_*.csv")),
	ignore_index=True
)

df_generated_stories.head()

### ChiSCor

In [ ]:
path_name = 'ChiSCor_master_df.csv'
df_chiscor = pd.read_csv(path_name, index_col=0)
df_chiscor = df_chiscor.rename(columns={'story_raw': 'story'})

### Reference corpora

In [ ]:
ref_standard_dep = Path('datasets/BasiScript/BS_dep_lexicon.csv')
ref_standard_uni = Path('datasets/BasiScript/BS_unigram_lexicon.csv')
ref_standard_bi = Path('datasets/BasiScript/BS_bigram_lexicon.csv')

In [ ]:
ref_spoken_b_csv = Path('datasets/CGN/CGN_pos_bigram.csv')
ref_spoken_u_csv  = Path('datasets/CGN/CGN_pos_unigram.csv')
ref_spoken_t_csv = Path('datasets/CGN/CGN_pos_trigram.csv')

In [ ]:
all_datasets = {'chiscor': df_chiscor}

## Story quality

Overview used metrics:
- Coherence
	- Local contextuality
- Grammaticality
	- Grammaticality
- Surprise
	- Creative perplexity
- Diversity
	- Lexical diversity: self-bleu
	- lexical diversity: moving mtld
- Complexity
	- Lexical complexity: unique words
	- Lexical complexity: Average word length
	- Syntactic complexity: avg. components
	- Syntactic complexity: dependency distance
	- Syntactic complexity: syntactic tree depth

In [ ]:
eval_f = EvaluationFramework(language='nl',
							  pos_unigram=ref_spoken_u_csv, 
							  pos_bigram=ref_spoken_b_csv,
							  pos_trigram=ref_spoken_t_csv,
							  ref_unigram=ref_standard_uni,
							  ref_bigram=ref_standard_bi,
							  ref_ling_constrained=ref_standard_dep,
							  embedding_model='jegormeister/bert-base-dutch-cased')

In [ ]:
# Surprise: creative perplexity
eval_f.add_pipe('creative_perplexity_dep')
# Local contextuality
eval_f.add_pipe("local_contextuality")
# Grammaticality
eval_f.add_pipe('grammaticality')
# Diversity (lexical) self-bleu
eval_f.add_pipe('self-bleu')
# Diversity (lexical) moving mtld
eval_f.add_pipe('lexical_diversity')
# Complexity (lexical) unique words
eval_f.add_pipe('unique-words')
# Complexity (lexical) average word length
eval_f.add_pipe('avg-word-length')
# Complexity (syntactic)  average components
eval_f.add_pipe('average_components')
# Complexity (syntactic) dependency distance
eval_f.add_pipe('dependency_distance')
# Complexity (syntactic) syntactic tree depth
eval_f.add_pipe('syntactic_depth')

In [ ]:
def load_or_run_eval(eval_f, dataset, column, path_name, *, run=False):
	if Path(path_name).exists() and not run:
		df = pd.read_csv(path_name)
		if 'lexical_diversity' in df.columns:
			df['lexical_diversity'] = df['lexical_diversity'].apply(ast.literal_eval)
			
		return df

	dataset = eval_f.run_pipeline_on_df(dataset, column)
	dataset.to_csv(path_name, index=False)
	return dataset

### Run evaluation for samples new dataset

In [ ]:
n_runs = 5
sample_size = 600
text_col = "story"
timestamp = datetime.now().strftime("%Y-%m-%d_%H-%M-%S")
out_dir = Path(f"results/new_dataset_results/{timestamp}")
out_dir.mkdir(parents=True, exist_ok=True)

final_summary = new_dataset_eval(df_generated_stories, n_runs, sample_size, text_col, out_dir, eval_f)
dataset_results = run_overall_dataset_statistics(df_generated_stories, out_dir, n_runs, sample_size, text_col)

### Run evaluation for ChiSCor

In [ ]:
temp = all_datasets
for name, data in temp.items():
	print(f"Current method: {name}")
	generated_eval_results = load_or_run_eval(eval_f, data, 'story', f'results/metric_results_without_newlines/eval_results_{name}.csv')
	all_datasets[name] = generated_eval_results

In [ ]:
def get_value_from_dictionary(x, key):
	new = []

	for item in x:
		new.append(item[key])
	return new

In [ ]:
metrics = ['creative_perplexity_dep', 'local_contextuality', 'grammaticality', 'self-bleu', 'lexical_diversity', 'unique-words', 'avg-word-length', 'average_components', 'dependency_distance', 'syntactic_depth']
story_metrics_means = {}
story_metrics_std = {}

for name in all_datasets.keys():
	story_metrics_means[name] = defaultdict(float)
	story_metrics_std[name] = defaultdict(float)

for name, data in all_datasets.items():
	for column in data.columns:
		if column in metrics:
			try:
				story_metrics_means[name][column] = data[column].mean()
				story_metrics_std[name][column] = data[column].std()
			except:
				x = get_value_from_dictionary(data[column], 'moving_mtld')
				story_metrics_means[name][column] = np.mean(np.array(x))
				story_metrics_std[name][column] = np.std(np.array(x))
				
df_story_means = pd.DataFrame.from_dict(story_metrics_means, orient="index")
df_story_means		

## SimLex for new dataset

In [ ]:
simlex999 = pd.read_csv('dataset_evaluation/modelasametric/SimLex-999-Dutch-final.txt', names=['lemma1','lemma2','SimLexScore','POS'], skiprows=1, sep='\t')
simlex999.drop(index=364, inplace=True) # weird pair, often misinterpreted due to different pos tags

In [ ]:
# initialise word2vec model
mincount = 10
trained_models = {}
n_runs = 5
sample_size = 600

for run in tqdm(range(1, n_runs + 1)):
	seed = 1000 + run	# reproducible but different each run
	sample = df_generated_stories.sample(n=sample_size, replace=False, random_state=seed).copy()

	model = train_and_save_w2v(sample['story'], f'dataset_evaluation/modelasametric/new_dataset/story_Word2Vec_run{run}_seed{seed}.kv', mincount=mincount)
	trained_models[f'{run}_{seed}'] = model

In [ ]:
cosine_similarities = {}
chosen_lemmas = {}
results = {}

for name, model in trained_models.items():
	df_filtered_simlex, filtered_simlex_list, filtered_simlex_tuple = select_invocab_simlex(simlex999, [model], sample=45)
	chosen_lemmas[name] = df_filtered_simlex
	cosine_similarities[name] = compute_similarity(filtered_simlex_tuple, model)
	r, p = spearmanr(cosine_similarities[name], df_filtered_simlex['SimLexScore'])
	results[(name, 'simlex')] = {"rho": r, "p": p}
	
results

## Radar plot

In [ ]:
rename_dict = {
    "creative_perplexity_dep": "Creative PPL.",
    "local_contextuality": "Local Contextuality",
    "grammaticality": "Grammaticality",
    "self-bleu": "Self-BLEU",
    "lexical_diversity": "Moving MTLD",
    "unique-words": "Amount of Unique Words",
    "avg-word-length": "Average Word Length",
    "average_components": "Average Components",
    "dependency_distance": "Dependency Distance",
    "syntactic_depth": "Syntactic Tree Depth",
}

higher_is_better = {
    "creative_perplexity_dep": True,
    "local_contextuality": True,
    "grammaticality": True,
    "self-bleu": False,
    "lexical_diversity": True,
    "unique-words": True,
    "avg-word-length": True,
    "average_components": True,
    "dependency_distance": True,
    "syntactic_depth": True,
}

In [ ]:
def radar_relative_change(
    df: pd.DataFrame,
    baseline: str,
    model: str,
    metric_rename: dict[str, str] | None = None,
    higher_is_better: dict[str, bool] | None = None,
    cap: float = 0.5,           # cap relative change at ±50%
    figsize=(7, 7),
    title: str | None = None,
    baseline_label: str | None = None,
    model_label: str | None = None,
    save_name: str | None = None,
):
    """
    df: rows=models, cols=metrics (raw values)
    baseline: index label of baseline row
    model: index label of model to compare
    """

    df_plot = df.copy()

    # Rename metrics early
    if metric_rename:
        df_plot = df_plot.rename(columns=metric_rename)

    # Extract rows
    base = df_plot.loc[baseline]
    mod = df_plot.loc[model]

    # Direction handling: ensure higher = better
    if higher_is_better:
        for m, hib in higher_is_better.items():
            m2 = metric_rename.get(m, m) if metric_rename else m
            if m2 in df_plot.columns and hib is False:
                base[m2] = -base[m2]
                mod[m2] = -mod[m2]

    # Relative change
    rel = (mod - base) / base.abs()

    # Clip extreme values
    rel = rel.clip(-cap, cap)

    # Map [-cap, cap] → [0, 1] for plotting
    values = (rel + cap) / (2 * cap)

    metrics = list(values.index)
    n = len(metrics)

    if n < 3:
        raise ValueError("Radar plots need at least 3 metrics.")

    # Angles
    angles = np.linspace(0, 2 * np.pi, n, endpoint=False).tolist()
    angles += angles[:1]

    # Close the loop
    plot_values = values.tolist()
    plot_values += plot_values[:1]

    # Baseline reference ring at 0.5
    baseline_ring = [0.5] * (n + 1)

    # Plot
    fig = plt.figure(figsize=figsize)
    ax = plt.subplot(111, polar=True)

    ax.set_theta_offset(np.pi / 2)
    ax.set_theta_direction(-1)

    ax.set_xticks(angles[:-1])
    ax.set_xticklabels(metrics)

    ax.set_ylim(0, 1)
    ax.set_yticks([0.0, 0.25, 0.5, 0.75, 1.0])
    ax.set_yticklabels([f"-{cap * 100}%", "", "", "", f"+{cap * 100}%"])

    # Baseline ring
    ax.plot(angles, baseline_ring, linestyle="--", linewidth=2, color="tab:blue", label=baseline_label)
    ax.fill(angles, baseline_ring, alpha=0.15, color="tab:blue")

    # Model
    ax.plot(angles, plot_values, linewidth=2, label=model_label, color="tab:orange")
    ax.fill(angles, plot_values, alpha=0.15, color="tab:orange")

    if title:
        ax.set_title(title, pad=20)
    
    ax.tick_params(axis='both', which='major', labelsize=12)
    
    # Remove the polar spine (outer circle)
    ax.spines["polar"].set_visible(False)

    for label in ax.get_xticklabels() + ax.get_yticklabels():
        label.set_fontweight('bold')

    ax.legend(loc="upper right", bbox_to_anchor=(1.25, 1.10), frameon=False, fontsize=12)
    plt.tight_layout()
    plt.savefig(save_name)

In [ ]:
df_transformed = (
    final_summary
    .set_index("metric")                 # make metric the index
    .rename_axis(None, axis=0) 
    .drop(columns=["std_over_runs"])      # drop std_over_runs
    .rename(columns={"mean_over_runs": "own_dataset"})
    .T                                   # transpose
)
df_transformed = df_transformed.drop(columns=["wbr_average", "index"])


In [ ]:
df_combined = pd.concat([df_transformed, df_story_means], axis=0)


In [ ]:
radar_relative_change(
    df_combined,
    baseline="chiscor",
    model="own_dataset",
    metric_rename=rename_dict,
    higher_is_better=higher_is_better,
    cap=0.5,
    baseline_label='ChiSCor',
    model_label='New Dataset',
    save_name='results/new_dataset_radar_plot.pdf'
)